# AICE Associate 변주 문제 v2 - 예상 오퍼 점수, 합격 확률 예측 (회귀)

### 시나리오

스타트업 '테크노솔루션'은 신규 엔지니어 채용 시 지원자의 잠재적 기여도를 예측하여 합격률을 최적화하고자 합니다. 회사는 과거 채용 데이터와 지원자의 배경 정보를 분석하여, 어떤 요인들이 높은 성과를 내는 인재를 선별하는 데 도움을 받고자 합니다. 이 모델은 비용 효율적인 채용 전략을 수립하고 인재 유치 경쟁력을 높이는 데 사용됩니다.

---

**[유의사항]**
- 답안은 각 문항 아래 표시된 `# (N) 여기에 ...` 칸에 작성하세요.
- **정답/해설은 이 노트북 가장 아래 `## 해설` 섹션에 모아뒀습니다.** 먼저 스스로 풀어본 뒤에 확인하세요.
- 이 노트북은 오리지널 창작 문제이며, 실제 AICE 샘플문항 원문을 복제하지 않습니다.
- 데이터 로더: `read_json` / 기본모델: `DecisionTreeRegressor` / 비교모델: `XGBRegressor` / 스케일러: `MinMaxScaler`

**[데이터 컬럼 설명]**

- predicted_offer_score : 예상 오퍼 점수, 합격 확률
- total_interview_score : 면접에서 받은 총점 (100점 만점)
- applicant_level : 지원자 레벨 (신입, 주니어, 시니어, 리드)
- region_code : 지원자의 거주 지역 코드 (KOREA-SEOUL, KOREA-BUSAN, KOREA-INCHEON 등 5개 지역 코드) (dim.csv 와 병합 키)
- years_experience : 피처 컬럼
- skill_match_score : 피처 컬럼
- interview_duration_days : 피처 컬럼
- education_level : 피처 컬럼
- applicant_id : 식별자(모델링에 불필요)
- (병합 후) dim_value : 지역별 평균 가중치

## 0. 데이터 준비

다음 문항을 풀기 전에 아래 코드를 실행하세요 (문제 데이터 2개 테이블을 생성합니다).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def synthesize_tables(seed=2023, n_rows=600):
    rng = np.random.default_rng(seed)
    n = n_rows

    base = rng.normal(loc=50, scale=15, size=n)
    outlier_idx = rng.choice(n, size=max(3, n // 50), replace=False)
    base[outlier_idx] += rng.choice([1, -1], size=len(outlier_idx)) * rng.uniform(80, 150, size=len(outlier_idx))

    main_df = pd.DataFrame({'total_interview_score': base.round(2)})

    cats = [f"Cat{i+1}" for i in range(rng.integers(3, 5))]
    main_df['applicant_level'] = rng.choice(cats, size=n)

    n_regions = rng.integers(5, 9)
    regions = [f"R{i+1:02d}" for i in range(n_regions)]
    main_df['region_code'] = rng.choice(regions, size=n)

    for col in ['years_experience', 'skill_match_score', 'interview_duration_days', 'education_level']:
        main_df[col] = rng.normal(0, 1, size=n).round(2)

    main_df['applicant_id'] = [f"ID{i:05d}" for i in range(n)]

    z = (base - base.mean()) / (base.std() + 1e-9)
    classes = ['예상 오퍼 점수', '합격 확률']
    if '회귀' == "분류":
        if False and len(classes) >= 3:
            bins = np.quantile(z, [1 / 3, 2 / 3])
            idx = np.digitize(z, bins)
            main_df['predicted_offer_score'] = [classes[i] for i in idx]
        else:
            prob = 1 / (1 + np.exp(-z))
            labels = (rng.random(n) < prob).astype(int)
            main_df['predicted_offer_score'] = np.where(labels == 1, classes[0], classes[-1])
    else:
        noise = rng.normal(0, 5, size=n)
        main_df['predicted_offer_score'] = (base * 1.5 + noise).round(2)

    for col in ['total_interview_score'] + ['years_experience', 'skill_match_score', 'interview_duration_days', 'education_level'][:1]:
        na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
        main_df.loc[na_idx, col] = np.nan

    main_df = main_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    dim_df = pd.DataFrame({
        'region_code': regions,
        "dim_value": rng.uniform(0.5, 2.0, size=n_regions).round(3),
    })
    return main_df, dim_df


main_df, dim_df = synthesize_tables()
main_df.to_json("data.json", index=False)
dim_df.to_csv("dim.csv", index=False)
print("데이터 저장 완료 - data.json:", main_df.shape, "/ dim.csv:", dim_df.shape)
main_df.head(4)

## <데이터 분석>

### 1. 라이브러리 임포트

pandas, numpy, matplotlib.pyplot, seaborn 을 각각 pd, np, plt, sns 별칭으로 임포트하세요.

In [ ]:
# (1) 여기에 답안코드를 작성하고 실행하세요



### 2. 데이터 로드 (read_csv/read_json)

`data.json` 를 `pd.read_json` 로 읽어 **my_data** 에, `dim.csv` 를 `pd.read_csv` 로 읽어 **dim_data** 에 각각 할당하세요.

In [ ]:
# (2) 여기에 답안코드를 작성하고 실행하세요



### 3. 결측치 확인

my_data 의 `total_interview_score` 컬럼에 결측치(NaN)가 몇 개 있는지 `isna().sum()` 으로 확인하세요. 몇 개입니까?

In [ ]:
# (3) 여기에 답을 입력하세요 (실행 불필요)



### 4. 데이터 병합 (pd.merge)

`region_code` 를 키로 my_data 와 dim_data 를 **left join** 하여 **data_merged** 에 저장하세요.

In [ ]:
# (4) 여기에 답안코드를 작성하고 실행하세요



### 5. 데이터 집계 (groupby)

`applicant_level` 별 `total_interview_score` 의 평균을 구해 **df_grp** 에 저장하세요.

In [ ]:
# (5) 여기에 답안코드를 작성하고 실행하세요



### 6. 시각화 (subplots)

지원자 레벨 (신입, 주니어, 시니어, 리드)(applicant_level) 분포의 countplot 과, 예상 오퍼 점수, 합격 확률 별 면접에서 받은 총점 (100점 만점)(total_interview_score) histplot 을 나란히 그리는 코드입니다.
빈칸 **(A)** 에 들어갈, 여러 그래프를 한 번에 그릴 때 쓰는 matplotlib 함수 이름은?

```python
fig, axes = plt.(A)(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='applicant_level', ax=axes[0])
sns.histplot(data=data_merged, x='total_interview_score', hue='predicted_offer_score', ax=axes[1])
plt.show()
```

In [ ]:
# (6) 여기에 답을 입력하세요 (실행 불필요)



### 7. 시각화 2

예상 오퍼 점수, 합격 확률 별 면접에서 받은 총점 (100점 만점)(total_interview_score) 분포를 seaborn boxplot 으로 시각화하세요.

In [ ]:
# (7) 여기에 답안코드를 작성하고 실행하세요



## <데이터 전처리>

### 8. 이상치 처리

IQR 기준(K=2.0)을 벗어나는 이상치 행을 제거하고, 식별자 컬럼도 삭제해서 **data_temp** 에 저장하는 코드입니다. 빈칸 **(A)** 에 들어갈, 행을 삭제할 때 쓰는 DataFrame 메서드 이름은?

```python
q1 = data_merged['total_interview_score'].quantile(0.25)
q3 = data_merged['total_interview_score'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 2.0 * iqr
upper_fence = q3 + 2.0 * iqr
data_temp = data_merged.(A)(data_merged[(data_merged['total_interview_score'] > upper_fence) | (data_merged['total_interview_score'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['applicant_id'])
data_temp = data_temp.reset_index(drop=True)
```

In [ ]:
# (8) 여기에 답을 입력하세요 (실행 불필요)



### 9. 결측치 처리

다음은 data_temp 의 결측치를 대표값으로 채우는 코드인데, 실행하면 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fill_value = data_temp['total_interview_score'].median()
data_na = data_temp.fillna({'total_interview_score': fill_value})
data_na['years_experience'] = data_na['years_experience'].fillna(data_na['years_experience'].median())
# (이 버전에는 의도한 것과 다른 결과를 내는 부분이 있습니다)
```

In [ ]:
# (9) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 10. 인코딩

`applicant_level` 는 원-핫 인코딩(get_dummies, drop_first=True), `region_code` 는 sklearn LabelEncoder 로 인코딩해서 data_preset 에 저장하세요.

In [ ]:
# (10) 여기에 답안코드를 작성하고 실행하세요



### 11. 데이터 분리

predicted_offer_score 을 y, 나머지를 X 로 삼아 train_test_split 으로 분리하세요.
- test_size=0.2, random_state=7
- 변수명: X_train, X_valid, y_train, y_valid

In [ ]:
# (11) 여기에 답안코드를 작성하고 실행하세요



### 12. 스케일링

MinMaxScaler 로 훈련/검증 데이터를 스케일링하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
# (이 버전에는 의도한 것과 다른 결과를 내는 부분이 있습니다)
```

In [ ]:
# (12) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 13. 스케일러 특성

MinMaxScaler 를 훈련 데이터에 적용하면, 결과 값의 분포는 이론적으로 어떤 특성을 가지게 됩니까?

In [ ]:
# (13) 여기에 답을 입력하세요 (실행 불필요)



## <AI 모델링>

### 14. 머신러닝 기본 (fit-predict)

DecisionTreeRegressor 로 모델을 하나 만들어 학습시키고, 검증데이터에 대한 예측값을 **pred_y** 에 저장하세요. (변수명: model, pred_y)

In [ ]:
# (14) 여기에 답안코드를 작성하고 실행하세요



### 15. GridSearch 모델링

DecisionTreeRegressor 와 XGBRegressor 를 GridSearchCV(cv=3)로 탐색하고 학습하세요.
- max_depth 후보: [3, 5, 7]
- XGBRegressor 의 n_estimators 후보: [50, 100, 200]
- 변수명: gs_a (베이스 모델), gs_b (비교 모델)

In [ ]:
# (15) 여기에 답안코드를 작성하고 실행하세요



### 16. GridSearch 결과 확인

위 GridSearch에서 XGBRegressor 의 n_estimators 후보는 [50, 100, 200] 였습니다. GridSearchCV가 고를 수 있는 값의 후보 중 '가장 큰 값'은 얼마인가요?

In [ ]:
# (16) 여기에 답을 입력하세요 (실행 불필요)



### 17. 변수중요도

XGBRegressor 의 변수중요도 Top 15개(정렬 ascending=False)를 뽑아 barh 차트로 시각화하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[:14]
plt.barh(fi['feature'], fi['importance'])
plt.show()
```

In [ ]:
# (17) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 18. 성능평가

검증데이터로 gs_a, gs_b 두 모델의 **mean_squared_error** 를 각각 계산해서 a_score, b_score 에 저장하세요.

In [ ]:
# (18) 여기에 답안코드를 작성하고 실행하세요



### 19. 모델 성능 비교

바로 위에서 계산한 a_score 와 b_score 를 비교했을 때, 어느 모델이 더 우수하다고 판단할 수 있습니까? (둘 중 하나를 실행 결과에 따라 답하세요)

In [ ]:
# (19) 여기에 답을 입력하세요 (실행 불필요)



### 20. 딥러닝 설계

다음은 딥러닝 모델을 설계하는 코드입니다. EarlyStopping 에서 '몇 epoch 동안 개선이 없으면 멈출지' 지정하는 파라미터 이름(빈칸 **(A)**)은? (은닉층 활성함수: relu, 출력층: linear/mse, BatchNormalization 포함, ModelCheckpoint 포함)

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
cb_list = [EarlyStopping(monitor='val_loss', (A)=10, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))
```

In [ ]:
# (20) 여기에 답안코드를 작성하고 실행하세요



### 21. 딥러닝 학습

위에서 설계한 model 을 batch_size=64, epochs=50 으로 학습하고 history 에 저장하세요 (callbacks=cb_list 사용).

In [ ]:
# (21) 여기에 답안코드를 작성하고 실행하세요



### 22. 학습곡선 시각화

history 를 이용해서 학습/검증 **mae** 변화를 한 그래프에 시각화하세요 (x축 라벨: epoch, 범례 위치: upper left, 범례 텍스트: train/val).

In [ ]:
# (22) 여기에 답안코드를 작성하고 실행하세요



### 23. 저장된 모델 재사용

ModelCheckpoint 로 저장된 `best_model.keras` 를 `load_model()` 로 다시 불러와서, 검증데이터에 대한 예측값을 **reload_pred** 에 저장하세요.

In [ ]:
# (23) 여기에 답안코드를 작성하고 실행하세요



---
## 해설

스스로 풀어본 뒤 아래에서 확인하세요. 문항 번호가 위 문제 번호와 일치합니다.

### 1번 해설 - 라이브러리 임포트 [코드작성]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

> import ... as ... 문법으로 널리 쓰이는 관례적 별칭을 지정합니다.

### 2번 해설 - 데이터 로드 (read_csv/read_json) [코드작성]

In [ ]:
my_data = pd.read_json('data.json')
dim_data = pd.read_csv('dim.csv')
my_data.head(4)

> pd.read_json() 로 파일 형식에 맞는 로더를 사용합니다.

### 3번 해설 - 결측치 확인 [결과값예측]

In [ ]:
18

> Series.isna().sum() 은 True(결측치)의 개수를 셉니다.

### 4번 해설 - 데이터 병합 (pd.merge) [코드작성]

In [ ]:
data_merged = pd.merge(my_data, dim_data, on='region_code', how='left')
data_merged.head(4)

> pd.merge(left, right, on=키, how='left') 는 왼쪽 테이블 기준으로 오른쪽 테이블을 결합합니다.

### 5번 해설 - 데이터 집계 (groupby) [코드작성]

In [ ]:
df_grp = data_merged.groupby('applicant_level')['total_interview_score'].mean()
df_grp

> groupby(기준컬럼)[대상컬럼].mean() 형태로 그룹별 집계를 구합니다.

### 6번 해설 - 시각화 (subplots) [빈칸채우기]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='applicant_level', ax=axes[0])
sns.histplot(data=data_merged, x='total_interview_score', hue='predicted_offer_score', ax=axes[1])
plt.show()

> plt.subplots() 는 nrows/ncols 로 여러 축(Axes)을 한 번에 만듭니다. 정답: subplots

### 7번 해설 - 시각화 2 [코드작성]

In [ ]:
sns.boxplot(data=data_merged, x='predicted_offer_score', y='total_interview_score')
plt.show()

> sns.boxplot(x=, y=) / sns.jointplot(x=, y=) 형태로 그립니다.

### 8번 해설 - 이상치 처리 [빈칸채우기]

In [ ]:
q1 = data_merged['total_interview_score'].quantile(0.25)
q3 = data_merged['total_interview_score'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 2.0 * iqr
upper_fence = q3 + 2.0 * iqr
data_temp = data_merged.drop(data_merged[(data_merged['total_interview_score'] > upper_fence) | (data_merged['total_interview_score'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['applicant_id'])
data_temp = data_temp.reset_index(drop=True)

> DataFrame.drop() 은 행(기본 axis=0) 또는 열(axis=1)을 삭제합니다. 정답: drop

### 9번 해설 - 결측치 처리 [오류정정]

In [ ]:
fill_value = data_temp['total_interview_score'].median()
data_na = data_temp.fillna({'total_interview_score': fill_value})
data_na['years_experience'] = data_na['years_experience'].fillna(data_na['years_experience'].median())

> [off_by_one] 오프바이원(Off-by-one) 오류: 인덱스 범위 초과, 반복/개수가 1 부족·초과되는 값 사용

### 10번 해설 - 인코딩 [코드작성]

In [ ]:
data_preset = pd.get_dummies(data=data_na, columns=['applicant_level'], drop_first=True)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_preset['region_code'] = le.fit_transform(data_preset['region_code'])
data_preset.info()

> 저-카디널리티는 원-핫, 코드성 범주는 라벨 인코딩을 흔히 사용합니다.

### 11번 해설 - 데이터 분리 [코드작성]

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop(['predicted_offer_score'], axis=1)
y = data_preset['predicted_offer_score']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=7)

> train_test_split(X, y, ...) 은 X_train, X_valid, y_train, y_valid 순서로 반환합니다.

### 12번 해설 - 스케일링 [오류정정]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

> [scope_reference] 스코프/참조 착각: fit/transform 순서, train/valid 뒤바뀜 등 데이터 흐름 실수

### 13번 해설 - 스케일러 특성 [결과값예측]

In [ ]:
'최솟값 0, 최댓값 1 사이 구간으로 정규화됩니다.'

> MinMaxScaler 의 정의에 따른 이론적 특성입니다.

### 14번 해설 - 머신러닝 기본 (fit-predict) [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=7)
model.fit(X_train_scaled, y_train)
pred_y = model.predict(X_valid_scaled)
pred_y[:5]

> import → model = 클래스() → model.fit(X_train, y_train) → pred_y = model.predict(X_valid) 4줄 템플릿입니다.

### 15번 해설 - GridSearch 모델링 [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

gs_a = GridSearchCV(DecisionTreeRegressor(random_state=7), {'max_depth':[3,5,7]}, cv=3)
gs_a.fit(X_train_scaled, y_train)

gs_b = GridSearchCV(XGBRegressor(random_state=7), {'n_estimators':[50, 100, 200], 'max_depth':[3,5,7]}, cv=3)
gs_b.fit(X_train_scaled, y_train)

> GridSearchCV(estimator, param_grid, cv=...).fit(X_train, y_train) 형태로 탐색합니다. XGBRegressor 는 sklearn 기본 앙상블 외에 XGBoost/LightGBM 계열일 수도 있습니다.

### 16번 해설 - GridSearch 결과 확인 [결과값예측]

In [ ]:
200

> 제시된 후보 중 GridSearch가 고를 수 있는 최댓값을 묻는 문항입니다.

### 17번 해설 - 변수중요도 [오류정정]

In [ ]:
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[:15]
plt.barh(fi['feature'], fi['importance'])
plt.show()

> [off_by_one] 오프바이원 오류가 발생했습니다. 원래 코드는 상위 15개의 특성을 선택하려고 했으나, 슬라이싱을 [:14]로 변경하여 실제로는 상위 14개의 특성만 선택하게 되었습니다. 이로 인해 기대했던 결과(상위 15개)와 실제 데이터의 개수가 일치하지 않게 됩니다.

### 18번 해설 - 성능평가 [코드작성]

In [ ]:
from sklearn.metrics import mean_squared_error

y_pred_a = gs_a.best_estimator_.predict(X_valid_scaled)
y_pred_b = gs_b.best_estimator_.predict(X_valid_scaled)

a_score = mean_squared_error(y_valid, y_pred_a)
b_score = mean_squared_error(y_valid, y_pred_b)
print(a_score, b_score)

> sklearn.metrics.mean_squared_error 에 (실제값, 예측값) 순서로 인자를 넣습니다.

### 19번 해설 - 모델 성능 비교 [결과값예측]

In [ ]:
'실행 결과에 따라 달라집니다: 값이 더 작은 쪽(오차이므로) 이 더 우수한 모델입니다. a_score, b_score 를 직접 비교해서 판단하세요.'

> 정확도/F1/ROC-AUC 등은 높을수록, MAE/MSE 등 오차 지표는 낮을수록 좋은 성능입니다.

### 20번 해설 - 딥러닝 설계 [빈칸채우기]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='linear')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
cb_list = [EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))

> EarlyStopping(patience=N) 은 N번 연속 개선이 없으면 학습을 멈춥니다. 정답: patience

### 21번 해설 - 딥러닝 학습 [코드작성]

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=64,
                    validation_data=(X_valid_scaled, y_valid), callbacks=cb_list)

> model.fit(X, y, epochs=, batch_size=, validation_data=, callbacks=) 형태입니다.

### 22번 해설 - 학습곡선 시각화 [코드작성]

In [ ]:
plt.plot(history.history['mae'])
plt.plot(history.history['val_mae'])
plt.title('Model mae')
plt.xlabel('epoch')
plt.ylabel('mae')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

> history.history[지표] 로 epoch별 기록을 꺼내 plt.plot() + plt.legend(loc=...) 으로 그립니다.

### 23번 해설 - 저장된 모델 재사용 [코드작성]

In [ ]:
from tensorflow.keras.models import load_model

saved_model = load_model('best_model.keras')
reload_pred = saved_model.predict(X_valid_scaled)
reload_pred[:5]

> 학습 없이도 저장된 가중치를 불러와(load_model) 바로 predict() 할 수 있습니다.